In [17]:
import pandas as pd

# --- settings ---
input_csv = "CRSP_2020_2025_cleaned.csv"
output_csv = "top_500_stocks_no_duplicates.csv"
top_x = 500

# -------------------------------------------------
# STEP 1: Load original data
# -------------------------------------------------
df = pd.read_csv(input_csv)

# Clean columns
df["Ticker"] = df["Ticker"].astype(str).str.strip()
df["DlyCalDt"] = pd.to_datetime(df["DlyCalDt"], errors="coerce")
df["DlyCap"] = pd.to_numeric(df["DlyCap"], errors="coerce")

# Drop bad rows
df = df.dropna(subset=["Ticker", "DlyCalDt", "DlyCap"])

print("Original number of rows:", len(df))

# -------------------------------------------------
# STEP 2: Remove duplicate Ticker/Date rows
# -------------------------------------------------
num_duplicates = df.duplicated(subset=["Ticker", "DlyCalDt"]).sum()
print("Number of duplicate Ticker/Date rows:", num_duplicates)

df_nodup = df.drop_duplicates(subset=["Ticker", "DlyCalDt"], keep="first").copy()

print("Rows after removing duplicates:", len(df_nodup))

# -------------------------------------------------
# STEP 3: Find top_x stocks on the global last date
# -------------------------------------------------
last_date = df_nodup["DlyCalDt"].max()
print("Global last date:", last_date.date())

last_date_df = df_nodup[df_nodup["DlyCalDt"] == last_date].copy()

top_tickers = (
    last_date_df.sort_values("DlyCap", ascending=False)
    .head(top_x)["Ticker"]
    .tolist()
)

print("Number of selected tickers:", len(top_tickers))
print("First 20 selected tickers:", top_tickers[:20])

# -------------------------------------------------
# STEP 4: Keep all rows for those tickers
# -------------------------------------------------
df_top = df_nodup[df_nodup["Ticker"].isin(top_tickers)].copy()
df_top = df_top.sort_values(["Ticker", "DlyCalDt"]).reset_index(drop=True)

# -------------------------------------------------
# STEP 5: Save result
# -------------------------------------------------
df_top.to_csv(output_csv, index=False)

print(f"Saved file to: {output_csv}")
print("Rows in output:", len(df_top))
print("Unique tickers in output:", df_top["Ticker"].nunique())

Original number of rows: 7255665
Number of duplicate Ticker/Date rows: 11023
Rows after removing duplicates: 7244642
Global last date: 2024-12-31
Number of selected tickers: 500
First 20 selected tickers: ['AAPL', 'NVDA', 'MSFT', 'AMZN', 'TSLA', 'GOOGL', 'AVGO', 'GOOG', 'LLY', 'WMT', 'JPM', 'SPY', 'IVV', 'VOO', 'V', 'MA', 'XOM', 'ORCL', 'UNH', 'VTI']
Saved file to: top_500_stocks_no_duplicates.csv
Rows in output: 622776
Unique tickers in output: 500


In [16]:
import pandas as pd

# --- settings ---
csv_path = "top_500_stocks_no_duplicates.csv"   # original dataset

# --- load ---
df = pd.read_csv(csv_path)

# clean columns
df["Ticker"] = df["Ticker"].astype(str).str.strip()
df["DlyCalDt"] = pd.to_datetime(df["DlyCalDt"], errors="coerce")

# count rows per ticker-date combination
ticker_date_counts = (
    df.groupby(["Ticker", "DlyCalDt"])
      .size()
      .reset_index(name="n_rows")
)

# keep only doubles
doubles = ticker_date_counts[ticker_date_counts["n_rows"] > 1].copy()

# count how many ticker names have at least one double date
tickers_with_doubles = doubles["Ticker"].nunique()

print("Number of tickers that have duplicate rows on at least one date:", tickers_with_doubles)
print("Total number of duplicated ticker-date combinations:", len(doubles))

print("\nFirst 50 ticker-date duplicates:")
print(doubles.head(50).to_string(index=False))

Number of tickers that have duplicate rows on at least one date: 0
Total number of duplicated ticker-date combinations: 0

First 50 ticker-date duplicates:
Empty DataFrame
Columns: [Ticker, DlyCalDt, n_rows]
Index: []


In [12]:
import pandas as pd

# Load data
df = pd.read_csv("top_500_stocks_no_duplicates.csv")

# Convert date column
df["DlyCalDt"] = pd.to_datetime(df["DlyCalDt"])

# Find earliest and latest date for each ticker
date_check = df.groupby("Ticker")["DlyCalDt"].agg(["min", "max"]).reset_index()
date_check = date_check.rename(columns={"min": "earliest_date", "max": "latest_date"})

print(date_check)

# Check if all tickers have the same earliest date
same_earliest = date_check["earliest_date"].nunique() == 1

# Check if all tickers have the same latest date
same_latest = date_check["latest_date"].nunique() == 1

print("\nSame earliest date for all tickers?", same_earliest)
print("Same latest date for all tickers?", same_latest)

if same_earliest:
    print("Common earliest date:", date_check["earliest_date"].iloc[0])

if same_latest:
    print("Common latest date:", date_check["latest_date"].iloc[0])

    

    Ticker earliest_date latest_date
0     AAPL    2020-01-02  2024-12-31
1     ABBV    2020-01-02  2024-12-31
2      ABG    2020-01-02  2024-12-31
3      ACN    2020-01-02  2024-12-31
4     ADBE    2020-01-02  2024-12-31
..     ...           ...         ...
495    XSD    2020-01-02  2024-12-31
496    XSW    2020-01-02  2024-12-31
497   ZBRA    2020-01-02  2024-12-31
498     ZS    2020-01-02  2024-12-31
499    ZTS    2020-01-02  2024-12-31

[500 rows x 3 columns]

Same earliest date for all tickers? False
Same latest date for all tickers? True
Common latest date: 2024-12-31 00:00:00


In [13]:
import pandas as pd

df = pd.read_csv("top_500_stocks_no_duplicates.csv")
df["DlyCalDt"] = pd.to_datetime(df["DlyCalDt"])

date_check = df.groupby("Ticker")["DlyCalDt"].agg(["min", "max"]).reset_index()
date_check = date_check.rename(columns={"min": "earliest_date", "max": "latest_date"})

common_earliest = date_check["earliest_date"].mode()[0]
common_latest = date_check["latest_date"].mode()[0]

different = date_check[
    (date_check["earliest_date"] != common_earliest) |
    (date_check["latest_date"] != common_latest)
]

print("All ticker date ranges:")
print(date_check)

print("\nTickers with different earliest/latest dates:")
print(different if not different.empty else "None")

All ticker date ranges:
    Ticker earliest_date latest_date
0     AAPL    2020-01-02  2024-12-31
1     ABBV    2020-01-02  2024-12-31
2      ABG    2020-01-02  2024-12-31
3      ACN    2020-01-02  2024-12-31
4     ADBE    2020-01-02  2024-12-31
..     ...           ...         ...
495    XSD    2020-01-02  2024-12-31
496    XSW    2020-01-02  2024-12-31
497   ZBRA    2020-01-02  2024-12-31
498     ZS    2020-01-02  2024-12-31
499    ZTS    2020-01-02  2024-12-31

[500 rows x 3 columns]

Tickers with different earliest/latest dates:
    Ticker earliest_date latest_date
20     AMR    2021-02-04  2024-12-31
43    AXON    2021-01-26  2024-12-31
86    CPAY    2024-03-25  2024-12-31
122     EG    2023-07-10  2024-12-31
124    ELV    2022-06-28  2024-12-31
288   MSGS    2020-04-20  2024-12-31
334   PIPR    2020-01-06  2024-12-31
370    RRX    2021-10-05  2024-12-31
408    TKO    2023-09-12  2024-12-31
417     TT    2020-03-02  2024-12-31
487    WTW    2022-01-10  2024-12-31


In [14]:
import pandas as pd

df = pd.read_csv("top_500_stocks_no_duplicates.csv")
print("Number of unique tickers:", df["Ticker"].nunique())

Number of unique tickers: 500
